In [30]:
import polars as pl
import numpy as np
import os, joblib, pickle
from datetime import date
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
)

In [ ]:
# ---- Load tech modeling table (primary) ----
tech_path = "../data/model_staging/tech_modeling_table.parquet"
df_tech = pl.read_parquet(tech_path)
print(f"Tech table shape: {df_tech.shape}")

# ---- Load fundamental features ----
fund_path = "../data/model_staging/fundamentalIndicators/modeling_fundamentals.parquet"
df_fund = pl.read_parquet(fund_path)

df_fund = df_fund.select([
    "symbol",
    pl.col("reportedDate").alias("earnings_date"),
    "eps_growth_qoq",
    "revenue_growth_qoq",
    "gross_margin", "gross_margin_qoq",
    "debt_to_equity", "debt_to_equity_qoq",
    "fcf_margin", "fcf_margin_qoq",
    "roe", "roe_qoq"
])

# ---- Load FinBERT sentiment ----
df_finbert = pl.read_parquet("../data/model_staging/finbert_tx_agg_weighted.parquet")
df_finbert = df_finbert.select([
    pl.col("symbol"),
    pl.col("reportedDate").alias("earnings_date"),
    "pos_prob",
    "neg_prob",
])
print(f"FinBERT shape: {df_finbert.shape}")

# ---- Load NZ news sentiment ----
df_nz = pl.read_parquet("../data/model_staging/nz_sentiment.parquet")
df_nz = df_nz.select([
    pl.col("symbol"),
    pl.col("reportedDate").alias("earnings_date"),
    "overall_sentiment_score_pre",
    "ticker_sentiment_score_pre",
    # "relevance_score_pre",
    "overall_sentiment_score_post",
    "ticker_sentiment_score_post",
    # "relevance_score_post",
])
print(f"NZ sentiment shape: {df_nz.shape}")

# ---- Merge all onto tech table ----
df_model = df_tech.join(df_fund, on=["symbol", "earnings_date"], how="left")
df_model = df_model.join(df_finbert, on=["symbol", "earnings_date"], how="left", suffix="_fb")
df_model = df_model.join(df_nz, on=["symbol", "earnings_date"], how="left", suffix="_nz")

print(f"Combined table shape: {df_model.shape}")
print(f"Target: {df_model['target_direction'].mean():.1%} positive")
print(f"\nNull counts for new sentiment cols:")
cols_added = [
    "pos_prob", "neg_prob",
    "overall_sentiment_score_pre", "ticker_sentiment_score_pre", # "relevance_score_pre",
    "overall_sentiment_score_post", "ticker_sentiment_score_post", # "relevance_score_post",
]

print(df_model.select(cols_added).null_count())

Tech table shape: (24048, 220)
FinBERT shape: (24294, 4)
NZ sentiment shape: (24294, 6)
Combined table shape: (24048, 236)
Target: 56.3% positive

Null counts for new sentiment cols:
shape: (1, 6)
┌──────────┬──────────┬───────────────────┬──────────────────┬──────────────────┬──────────────────┐
│ pos_prob ┆ neg_prob ┆ overall_sentiment ┆ ticker_sentiment ┆ overall_sentimen ┆ ticker_sentiment │
│ ---      ┆ ---      ┆ _score_pre        ┆ _score_pre       ┆ t_score_post     ┆ _score_post      │
│ u32      ┆ u32      ┆ ---               ┆ ---              ┆ ---              ┆ ---              │
│          ┆          ┆ u32               ┆ u32              ┆ u32              ┆ u32              │
╞══════════╪══════════╪═══════════════════╪══════════════════╪══════════════════╪══════════════════╡
│ 0        ┆ 0        ┆ 0                 ┆ 0                ┆ 0                ┆ 0                │
└──────────┴──────────┴───────────────────┴──────────────────┴──────────────────┴───────────────

In [32]:
print(df_model.select(cols_added))

shape: (24_048, 6)
┌──────────┬──────────┬───────────────────┬──────────────────┬──────────────────┬──────────────────┐
│ pos_prob ┆ neg_prob ┆ overall_sentiment ┆ ticker_sentiment ┆ overall_sentimen ┆ ticker_sentiment │
│ ---      ┆ ---      ┆ _score_pre        ┆ _score_pre       ┆ t_score_post     ┆ _score_post      │
│ f64      ┆ f64      ┆ ---               ┆ ---              ┆ ---              ┆ ---              │
│          ┆          ┆ f64               ┆ f64              ┆ f64              ┆ f64              │
╞══════════╪══════════╪═══════════════════╪══════════════════╪══════════════════╪══════════════════╡
│ 0.326023 ┆ 0.515066 ┆ 0.201455          ┆ 0.286114         ┆ 0.25455          ┆ 0.231505         │
│ 0.334268 ┆ 0.550935 ┆ 0.114912          ┆ -0.007841        ┆ 0.072336         ┆ 0.087317         │
│ 0.385011 ┆ 0.581205 ┆ 0.1387405         ┆ 0.055516         ┆ 0.272668         ┆ 0.267144         │
│ 0.217921 ┆ 0.646675 ┆ 0.271742          ┆ 0.273547         ┆ 0.271742 

In [33]:
# ---- Sector encoding from FinBERT data ----
df_sector_raw = pl.read_parquet("../data/model_staging/finbert_tx_agg_weighted.parquet")
df_sector = df_sector_raw.select(["symbol", "sector"]).unique()

# Fill null sectors
df_sector = df_sector.with_columns(
    pl.col("sector").fill_null("Unknown")
)

# One-hot encode
sectors = sorted(df_sector["sector"].unique().to_list())
df_sector = df_sector.with_columns([
    (pl.col("sector") == s).cast(pl.Int8).alias(f"sector_{s.replace(' ', '_')}")
    for s in sectors
]).drop("sector")

print(f"Sectors: {sectors}")
print(f"Unique symbols in sector map: {df_sector.shape[0]}")

# Join to df_model
df_model = df_model.join(df_sector, on="symbol", how="left")

sector_cols = [c for c in df_model.columns if c.startswith("sector_")]
print(f"Shape after sector join: {df_model.shape}")
print(f"Sector columns: {sector_cols}")

Sectors: ['Basic Materials', 'Communication Services', 'Consumer Cyclical', 'Consumer Defensive', 'Energy', 'Financial Services', 'Healthcare', 'Industrials', 'Real Estate', 'Technology', 'Unknown', 'Utilities']
Unique symbols in sector map: 503
Shape after sector join: (24048, 248)
Sector columns: ['sector_Basic_Materials', 'sector_Communication_Services', 'sector_Consumer_Cyclical', 'sector_Consumer_Defensive', 'sector_Energy', 'sector_Financial_Services', 'sector_Healthcare', 'sector_Industrials', 'sector_Real_Estate', 'sector_Technology', 'sector_Unknown', 'sector_Utilities']


In [34]:
Train_Window = 7
Folds = [
    (date(y - Train_Window, 1, 1), date(y, 1, 1), date(y + 1, 1, 1))
    for y in range(2021, 2026)
]

feature_cols = [c for c in df_model.columns
                if c not in ["symbol", "earnings_date", "target_return", "target_direction"]]

print(f"Total features: {len(feature_cols)}")

folds_data = []

for fold_num, (train_start, test_start, test_end) in enumerate(Folds, 1):
    train = df_model.filter(
        (pl.col("earnings_date") >= train_start) & (pl.col("earnings_date") < test_start)
    )
    test = df_model.filter(
        (pl.col("earnings_date") >= test_start) & (pl.col("earnings_date") < test_end)
    )

    X_train = train.select(feature_cols).to_numpy()
    X_test = test.select(feature_cols).to_numpy()

    # inf -> NaN -> median impute
    X_train = np.where(np.isinf(X_train), np.nan, X_train)
    X_test = np.where(np.isinf(X_test), np.nan, X_test)
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    # scaled version for linear models
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    folds_data.append({
        "fold_num": fold_num,
        "train_years": f"{train_start.year}-{test_start.year - 1}",
        "test_year": test_start.year,
        "X_train": X_train, "X_test": X_test,
        "X_train_sc": X_train_sc, "X_test_sc": X_test_sc,
        "y_train_dir": train["target_direction"].to_numpy(),
        "y_test_dir": test["target_direction"].to_numpy(),
        "y_train_ret": train["target_return"].to_numpy(),
        "y_test_ret": test["target_return"].to_numpy(),
    })

    print(f"Fold {fold_num}: train [{train_start.year}-{test_start.year-1}] "
          f"({len(X_train):,}) -> test [{test_start.year}] ({len(X_test):,})")

Total features: 244
Fold 1: train [2014-2020] (13,169) -> test [2021] (1,970)
Fold 2: train [2015-2021] (13,313) -> test [2022] (1,975)
Fold 3: train [2016-2022] (13,439) -> test [2023] (1,984)
Fold 4: train [2017-2023] (13,560) -> test [2024] (2,000)
Fold 5: train [2018-2024] (13,671) -> test [2025] (2,008)


In [35]:
np.random.seed(42)

b0_clf = {"fold_acc": [], "preds": [], "true": []}
b0_reg = {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}

for f in folds_data:
    preds_dir = np.random.randint(0, 2, size=len(f["y_test_dir"]))
    acc = accuracy_score(f["y_test_dir"], preds_dir)
    b0_clf["fold_acc"].append(acc)
    b0_clf["preds"].extend(preds_dir)
    b0_clf["true"].extend(f["y_test_dir"])

    preds_ret = np.full(len(f["y_test_ret"]), f["y_train_ret"].mean())
    b0_reg["fold_mae"].append(mean_absolute_error(f["y_test_ret"], preds_ret))
    b0_reg["fold_rmse"].append(np.sqrt(mean_squared_error(f["y_test_ret"], preds_ret)))
    b0_reg["preds"].extend(preds_ret)
    b0_reg["true"].extend(f["y_test_ret"])

    print(f"Fold {f['fold_num']}: DA={acc:.4f}")

print(f"\nAvg DA:   {np.mean(b0_clf['fold_acc']):.4f}")
print(f"Avg MAE:  {np.mean(b0_reg['fold_mae']):.4f}")
print(f"Avg RMSE: {np.mean(b0_reg['fold_rmse']):.4f}")

Fold 1: DA=0.5036
Fold 2: DA=0.5084
Fold 3: DA=0.5121
Fold 4: DA=0.4980
Fold 5: DA=0.5075

Avg DA:   0.5059
Avg MAE:  0.0419
Avg RMSE: 0.0576


In [36]:
lr_clf = {"fold_acc": [], "preds": [], "probs": [], "true": []}
lr_model = LogisticRegression(max_iter=1000, class_weight="balanced")

for f in folds_data:
    lr_model.fit(f["X_train_sc"], f["y_train_dir"])
    preds = lr_model.predict(f["X_test_sc"])
    probs = lr_model.predict_proba(f["X_test_sc"])[:, 1]

    acc = accuracy_score(f["y_test_dir"], preds)
    lr_clf["fold_acc"].append(acc)
    lr_clf["preds"].extend(preds)
    lr_clf["probs"].extend(probs)
    lr_clf["true"].extend(f["y_test_dir"])

    print(f"Fold {f['fold_num']} [{f['test_year']}]: DA={acc:.4f}")

print(f"\nAvg DA:     {np.mean(lr_clf['fold_acc']):.4f}")
print(f"Pooled F1:  {f1_score(np.array(lr_clf['true']), np.array(lr_clf['preds']), average='weighted'):.4f}")
print(f"Pooled AUC: {roc_auc_score(np.array(lr_clf['true']), np.array(lr_clf['probs'])):.4f}")

Fold 1 [2021]: DA=0.5391
Fold 2 [2022]: DA=0.5352
Fold 3 [2023]: DA=0.5292
Fold 4 [2024]: DA=0.4795
Fold 5 [2025]: DA=0.5005

Avg DA:     0.5167
Pooled F1:  0.5185
Pooled AUC: 0.5180


In [37]:
linreg = {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}
linreg_model = LinearRegression()

for f in folds_data:
    linreg_model.fit(f["X_train_sc"], f["y_train_ret"])
    preds = linreg_model.predict(f["X_test_sc"])

    mae = mean_absolute_error(f["y_test_ret"], preds)
    rmse = np.sqrt(mean_squared_error(f["y_test_ret"], preds))
    linreg["fold_mae"].append(mae)
    linreg["fold_rmse"].append(rmse)
    linreg["preds"].extend(preds)
    linreg["true"].extend(f["y_test_ret"])

    print(f"Fold {f['fold_num']} [{f['test_year']}]: MAE={mae:.4f}  RMSE={rmse:.4f}")

print(f"\nAvg MAE:   {np.mean(linreg['fold_mae']):.4f}")
print(f"Avg RMSE:  {np.mean(linreg['fold_rmse']):.4f}")
print(f"Pooled R²: {r2_score(np.array(linreg['true']), np.array(linreg['preds'])):.4f}")

Fold 1 [2021]: MAE=0.0399  RMSE=0.0553
Fold 2 [2022]: MAE=0.0524  RMSE=0.0718
Fold 3 [2023]: MAE=0.0394  RMSE=0.0533
Fold 4 [2024]: MAE=0.0421  RMSE=0.1033
Fold 5 [2025]: MAE=0.0511  RMSE=0.2547

Avg MAE:   0.0450
Avg RMSE:  0.1077
Pooled R²: -4.2185


In [38]:
rf_clf = {"fold_acc": [], "preds": [], "probs": [], "true": []}
rf_clf_model = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight="balanced", random_state=42, n_jobs=-1)

for f in folds_data:
    rf_clf_model.fit(f["X_train"], f["y_train_dir"])
    preds = rf_clf_model.predict(f["X_test"])
    probs = rf_clf_model.predict_proba(f["X_test"])[:, 1]

    acc = accuracy_score(f["y_test_dir"], preds)
    rf_clf["fold_acc"].append(acc)
    rf_clf["preds"].extend(preds)
    rf_clf["probs"].extend(probs)
    rf_clf["true"].extend(f["y_test_dir"])

    print(f"Fold {f['fold_num']} [{f['test_year']}]: DA={acc:.4f}")

print(f"\nAvg DA:     {np.mean(rf_clf['fold_acc']):.4f}")
print(f"Pooled F1:  {f1_score(np.array(rf_clf['true']), np.array(rf_clf['preds']), average='weighted'):.4f}")
print(f"Pooled AUC: {roc_auc_score(np.array(rf_clf['true']), np.array(rf_clf['probs'])):.4f}")

Fold 1 [2021]: DA=0.5472
Fold 2 [2022]: DA=0.5641
Fold 3 [2023]: DA=0.5252
Fold 4 [2024]: DA=0.5300
Fold 5 [2025]: DA=0.5179

Avg DA:     0.5369
Pooled F1:  0.5189
Pooled AUC: 0.5228


In [39]:
rf_reg = {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}
rf_reg_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)

for f in folds_data:
    rf_reg_model.fit(f["X_train"], f["y_train_ret"])
    preds = rf_reg_model.predict(f["X_test"])

    mae = mean_absolute_error(f["y_test_ret"], preds)
    rmse = np.sqrt(mean_squared_error(f["y_test_ret"], preds))
    rf_reg["fold_mae"].append(mae)
    rf_reg["fold_rmse"].append(rmse)
    rf_reg["preds"].extend(preds)
    rf_reg["true"].extend(f["y_test_ret"])

    print(f"Fold {f['fold_num']} [{f['test_year']}]: MAE={mae:.4f}  RMSE={rmse:.4f}")

print(f"\nAvg MAE:   {np.mean(rf_reg['fold_mae']):.4f}")
print(f"Avg RMSE:  {np.mean(rf_reg['fold_rmse']):.4f}")
print(f"Pooled R²: {r2_score(np.array(rf_reg['true']), np.array(rf_reg['preds'])):.4f}")

Fold 1 [2021]: MAE=0.0388  RMSE=0.0532
Fold 2 [2022]: MAE=0.0515  RMSE=0.0712
Fold 3 [2023]: MAE=0.0376  RMSE=0.0511
Fold 4 [2024]: MAE=0.0392  RMSE=0.0557
Fold 5 [2025]: MAE=0.0441  RMSE=0.0627

Avg MAE:   0.0423
Avg RMSE:  0.0588
Pooled R²: -0.0495


In [40]:
xgb_clf = {"fold_acc": [], "preds": [], "probs": [], "true": []}
xgb_clf_model = XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=42)

for f in folds_data:
    xgb_clf_model.fit(f["X_train"], f["y_train_dir"])
    preds = xgb_clf_model.predict(f["X_test"])
    probs = xgb_clf_model.predict_proba(f["X_test"])[:, 1]

    acc = accuracy_score(f["y_test_dir"], preds)
    xgb_clf["fold_acc"].append(acc)
    xgb_clf["preds"].extend(preds)
    xgb_clf["probs"].extend(probs)
    xgb_clf["true"].extend(f["y_test_dir"])

    print(f"Fold {f['fold_num']} [{f['test_year']}]: DA={acc:.4f}")

print(f"\nAvg DA:     {np.mean(xgb_clf['fold_acc']):.4f}")
print(f"Pooled F1:  {f1_score(np.array(xgb_clf['true']), np.array(xgb_clf['preds']), average='weighted'):.4f}")
print(f"Pooled AUC: {roc_auc_score(np.array(xgb_clf['true']), np.array(xgb_clf['probs'])):.4f}")

Fold 1 [2021]: DA=0.5228
Fold 2 [2022]: DA=0.5377
Fold 3 [2023]: DA=0.5307
Fold 4 [2024]: DA=0.5315
Fold 5 [2025]: DA=0.5129

Avg DA:     0.5272
Pooled F1:  0.5178
Pooled AUC: 0.5139


In [41]:
xgb_reg = {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}
xgb_reg_model = XGBRegressor(n_estimators=200, random_state=42)

for f in folds_data:
    xgb_reg_model.fit(f["X_train"], f["y_train_ret"])
    preds = xgb_reg_model.predict(f["X_test"])

    mae = mean_absolute_error(f["y_test_ret"], preds)
    rmse = np.sqrt(mean_squared_error(f["y_test_ret"], preds))
    xgb_reg["fold_mae"].append(mae)
    xgb_reg["fold_rmse"].append(rmse)
    xgb_reg["preds"].extend(preds)
    xgb_reg["true"].extend(f["y_test_ret"])

    print(f"Fold {f['fold_num']} [{f['test_year']}]: MAE={mae:.4f}  RMSE={rmse:.4f}")

print(f"\nAvg MAE:   {np.mean(xgb_reg['fold_mae']):.4f}")
print(f"Avg RMSE:  {np.mean(xgb_reg['fold_rmse']):.4f}")
print(f"Pooled R²: {r2_score(np.array(xgb_reg['true']), np.array(xgb_reg['preds'])):.4f}")

Fold 1 [2021]: MAE=0.0424  RMSE=0.0576
Fold 2 [2022]: MAE=0.0559  RMSE=0.0759
Fold 3 [2023]: MAE=0.0406  RMSE=0.0546
Fold 4 [2024]: MAE=0.0424  RMSE=0.0584
Fold 5 [2025]: MAE=0.0494  RMSE=0.0681

Avg MAE:   0.0461
Avg RMSE:  0.0629
Pooled R²: -0.2029


In [42]:
print("=== Direction (Classification) ===")
print(f"{'Model':<25} {'Avg DA%':>10} {'Pooled F1':>10} {'Prec':>8} {'Recall':>8} {'AUC':>8}")
print("-" * 75)

for name, r in [("B0 Random", b0_clf), ("Logistic Regression", lr_clf),
                ("Random Forest", rf_clf), ("XGBoost", xgb_clf)]:
    true = np.array(r["true"])
    preds = np.array(r["preds"])
    avg_acc = np.mean(r["fold_acc"])
    f1 = f1_score(true, preds, average="weighted")
    prec = precision_score(true, preds, average="weighted")
    rec = recall_score(true, preds, average="weighted")
    auc_str = f"{roc_auc_score(true, np.array(r['probs'])):>8.4f}" if r.get("probs") else f"{'—':>8}"
    print(f"{name:<25} {avg_acc:>10.4f} {f1:>10.4f} {prec:>8.4f} {rec:>8.4f} {auc_str}")

print(f"\n=== Magnitude (Regression) ===")
print(f"{'Model':<25} {'Avg MAE':>10} {'Avg RMSE':>10} {'Pooled R²':>10}")
print("-" * 60)

for name, r in [("B0 Mean", b0_reg), ("Linear Regression", linreg),
                ("Random Forest", rf_reg), ("XGBoost", xgb_reg)]:
    true = np.array(r["true"])
    preds = np.array(r["preds"])
    print(f"{name:<25} {np.mean(r['fold_mae']):>10.4f} {np.mean(r['fold_rmse']):>10.4f} {r2_score(true, preds):>10.4f}")

=== Direction (Classification) ===
Model                        Avg DA%  Pooled F1     Prec   Recall      AUC
---------------------------------------------------------------------------
B0 Random                     0.5059     0.5081   0.5152   0.5059        —
Logistic Regression           0.5167     0.5185   0.5224   0.5166   0.5180
Random Forest                 0.5369     0.5189   0.5201   0.5368   0.5228
XGBoost                       0.5272     0.5178   0.5160   0.5271   0.5139

=== Magnitude (Regression) ===
Model                        Avg MAE   Avg RMSE  Pooled R²
------------------------------------------------------------
B0 Mean                       0.0419     0.0576    -0.0020
Linear Regression             0.0450     0.1077    -4.2185
Random Forest                 0.0423     0.0588    -0.0495
XGBoost                       0.0461     0.0629    -0.2029


In [43]:
os.makedirs("models", exist_ok=True)

for name, model in [("lr_clf", lr_model), ("linreg", linreg_model),
                     ("rf_clf", rf_clf_model), ("rf_reg", rf_reg_model),
                     ("xgb_clf", xgb_clf_model), ("xgb_reg", xgb_reg_model)]:
    joblib.dump(model, f"models/b3_{name}.pkl")
    print(f"Saved models/b3_{name}.pkl")

with open("models/b3_results.pkl", "wb") as f:
    pickle.dump({
        "clf": {"B0 Random": b0_clf, "Logistic Regression": lr_clf,
                "Random Forest": rf_clf, "XGBoost": xgb_clf},
        "reg": {"B0 Mean": b0_reg, "Linear Regression": linreg,
                "Random Forest": rf_reg, "XGBoost": xgb_reg},
    }, f)
print("Saved models/b3_results.pkl")

Saved models/b3_lr_clf.pkl
Saved models/b3_linreg.pkl
Saved models/b3_rf_clf.pkl
Saved models/b3_rf_reg.pkl
Saved models/b3_xgb_clf.pkl
Saved models/b3_xgb_reg.pkl
Saved models/b3_results.pkl
